In [ ]:
import requests
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

In [ ]:

# ==============================
# 2. Endpoint e parametros de busca
# ==============================
BASE_URL = "https://pncp.gov.br/api/consulta/v1/contratos"

PARAMS_BASE = {
    "dataInicial": "20260101",
    "dataFinal": "20260411",
    "tamanhoPagina": 500
}

MAX_WORKERS = 10   
RETRIES = 3       

# ==============================
# 3. Função com retry
# ==============================
def fetch_page(page):
    params = PARAMS_BASE.copy()
    params["pagina"] = page

    for attempt in range(RETRIES):
        try:
            response = requests.get(BASE_URL, params=params, timeout=30)

            if response.status_code == 200:
                return response.json().get("data", [])
            
            else:
                print(f"Erro HTTP {response.status_code} na página {page}")

        except requests.exceptions.RequestException:
            print(f"Erro de conexão na página {page} (tentativa {attempt+1})")

        time.sleep(1)  # espera antes de tentar novamente

    return []  # se falhar tudo

# ==============================
# 4. Descobrir total de páginas
# ==============================
print("Consultando total de páginas...")

first_response = requests.get(BASE_URL, params={**PARAMS_BASE, "pagina": 1})
data = first_response.json()

total_paginas = data.get("totalPaginas", 1)

print(f"Total de páginas: {total_paginas}")

# ==============================
# 5. Coleta paralela
# ==============================
all_data = []

print("Baixando dados em paralelo...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(fetch_page, p): p for p in range(1, total_paginas + 1)}

    for future in tqdm(as_completed(futures), total=total_paginas):
        result = future.result()
        all_data.extend(result)

print(f"Total bruto coletado: {len(all_data)}")


In [ ]:
# ==============================
# 6. Normalizar JSON
# ==============================
df = pd.json_normalize(all_data)

print("Colunas disponíveis:")
print(df.columns.tolist())

# ==============================
# 7. Filtrar Ceará (CE)
# ==============================
if "unidadeOrgao.ufSigla" in df.columns:
    df_ce = df[df["unidadeOrgao.ufSigla"] == "CE"]
else:
    print("Coluna de UF não encontrada!")
    df_ce = df

print(f"Total CE: {len(df_ce)}")

for col in ["dataAssinatura", "dataVigenciaInicio", "dataVigenciaFim"]:
    if col in df_ce.columns:
        df_ce[col] = pd.to_datetime(df_ce[col], errors='coerce')

# ==============================
# 💾 9. Salvar CSV
# ==============================
output_file = "contratos_ceara.csv"

df_ce.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"Arquivo salvo como: {output_file}")